   # Migrate raw CSV to Parquet (one-off)

   Version 1 of the ingestion saved raw files as CSV. This notebook converted
   all 48 monthly files (2015-01 to 2018-12) to Parquet, with every column kept
   as text, and checked the row count of each file. The CSV files were then
   removed from S3.

   Run once. New months are written as Parquet directly by `loans_extract_job.py`.

In [ ]:
import os
from io import BytesIO

import boto3
import pandas as pd
from dotenv import load_dotenv, find_dotenv

print(load_dotenv(find_dotenv(usecwd=True)))

In [ ]:
AWS_CONFIG = {
    "aws_access_key_id": os.getenv("AWS_ACCESS_KEY"),
    "aws_secret_access_key": os.getenv("SECRET_KEY"),
    "region_name": os.getenv("REGION"),
}
s3 = boto3.client("s3", **AWS_CONFIG)

In [ ]:
BUCKET = "lending-data-de-project"
CSV_PREFIX = "lending-data-monthly/raw/"

response = s3.list_objects_v2(Bucket=BUCKET, Prefix=CSV_PREFIX)
csv_keys = [obj["Key"] for obj in response["Contents"] if obj["Key"].endswith(".csv")]
csv_keys = sorted(csv_keys)

print(len(csv_keys))
print(csv_keys[:3])

In [ ]:
first_key = csv_keys[0]

obj = s3.get_object(Bucket=BUCKET, Key=first_key)
df = pd.read_csv(BytesIO(obj["Body"].read()), dtype=str)

print(first_key)
print(df.shape)
df.head()

In [ ]:
from datetime import date
from pathlib import Path

INGEST_DATE = date.today().isoformat()                         # e.g. "2026-11-08"
LOCAL_RAW = Path("raw_parquet") / "loans" / f"ingest_date={INGEST_DATE}"
LOCAL_RAW.mkdir(parents=True, exist_ok=True)

file_name = first_key.split("/")[-1].replace(".csv", ".parquet")   # loans_2015-01.parquet
out_file = LOCAL_RAW / file_name

df.to_parquet(out_file, index=False)
print(out_file)

In [ ]:
check = pd.read_parquet(out_file)

print(check.shape)                                   # should match df.shape
print(check.shape == df.shape)                       # True

csv_mb = obj["ContentLength"] / 1024 / 1024
pq_mb = out_file.stat().st_size / 1024 / 1024
print(f"CSV {csv_mb:.1f} MB -> Parquet {pq_mb:.1f} MB")